In [ ]:
#--- Regression Analysis ---

import pandas as pd
import statsmodels.api as sm

# --- Load data, Excel should be changed based on grade level ---
df = pd.read_excel("8th grade .xlsx")
df.columns = df.columns.str.strip()

ach_map = {"Unsatisfactory": 1,
           "Approaching Basic": 2,
           "Basic": 3,
           "Mastery": 4}
rw_map = {"Weak": 1,
          "Moderate": 2,
          "Strong": 3}

df["Math_A_num"]   = df["Math Achievement"].map(ach_map)
df["ELA_A_num"]    = df["ELA Achievement"].map(ach_map)
df["ELA_Read_num"] = df["ELA Reading Performance"].map(rw_map)
df["ELA_Write_num"] = df["ELA Writing Performance"].map(rw_map)

# New vs Returning
if "New/Returning" in df.columns:
    df["New"] = (df["New/Returning"].astype(str).str.strip() == "New").astype(int)
else:
    df["New"] = (df["New/Returning "].astype(str).str.strip() == "New").astype(int)

# SPED vs Regular
df["SPED"] = (df["EducationClassificationSummary"]
              .astype(str).str.strip() == "Special").astype(int)

# Economically Disadvantaged vs Not
df["ED"] = (df["LAP Economically Disadvantaged"]
            .astype(str).str.strip() == "Economically Disadvantaged").astype(int)

# ----------------- MODEL 1: Predict Math Achievement ----------------- #
predictors_math = ["ELA_A_num", "ELA_Read_num", "ELA_Write_num", "SPED", "ED", "New"]
model1_df = df[predictors_math + ["Math_A_num"]].dropna()

X1 = sm.add_constant(model1_df[predictors_math])
y1 = model1_df["Math_A_num"]

model1 = sm.OLS(y1, X1).fit()
print(model1.summary())

# ----------------- MODEL 2: Predict ELA Achievement ----------------- #
predictors_ela = ["Math_A_num", "ELA_Read_num", "ELA_Write_num", "SPED", "ED", "New"]
model2_df = df[predictors_ela + ["ELA_A_num"]].dropna()

X2 = sm.add_constant(model2_df[predictors_ela])
y2 = model2_df["ELA_A_num"]

model2 = sm.OLS(y2, X2).fit()
print(model2.summary())

# If you want a quick look at just betas + p-values:
print("\nModel 1 Coefficients (Math DV):")
print(model1.params)
print("\nModel 1 p-values:")
print(model1.pvalues)

print("\nModel 2 Coefficients (ELA DV):")
print(model2.params)
print("\nModel 2 p-values:")
print(model2.pvalues)


In [ ]:
# --- Correlation Analysis ---
import pandas as pd

# --- Load data, Excel should be changed based on grade level---
df = pd.read_excel("8th grade .xlsx")
df.columns = df.columns.str.strip()

ach_map = {"Unsatisfactory": 1,
           "Approaching Basic": 2,
           "Basic": 3,
           "Mastery": 4}

rw_map = {"Weak": 1,
          "Moderate": 2,
          "Strong": 3}

df["Math_A_num"] = df["Math Achievement"].map(ach_map)
df["ELA_A_num"] = df["ELA Achievement"].map(ach_map)
df["ELA_Read_num"] = df["ELA Reading Performance"].map(rw_map)
df["ELA_Write_num"] = df["ELA Writing Performance"].map(rw_map)


corr_vars = ["Math_A_num", "ELA_A_num", "ELA_Read_num", "ELA_Write_num"]

corr_matrix = df[corr_vars].corr()

print(corr_matrix)


In [ ]:
# --- ANOVA Analyis ---
import pandas as pd
from scipy.stats import f_oneway

# --- Load data, Excel should be chnaged based on grade level ---
df = pd.read_excel("8th grade .xlsx")
df.columns = df.columns.str.strip()  # clean extra spaces

ach_map = {"Unsatisfactory": 1,
           "Approaching Basic": 2,
           "Basic": 3,
           "Mastery": 4}

rw_map = {"Weak": 1,
          "Moderate": 2,
          "Strong": 3}

df["Math_A_num"] = df["Math Achievement"].map(ach_map)
df["ELA_A_num"] = df["ELA Achievement"].map(ach_map)
df["ELA_Read_num"] = df["ELA Reading Performance"].map(rw_map)
df["ELA_Write_num"] = df["ELA Writing Performance"].map(rw_map)

if "New/Returning" in df.columns:
    df["NewRet"] = df["New/Returning"].astype(str).str.strip()
else:
    df["NewRet"] = df["New/Returning "].astype(str).str.strip()

df["Edu"] = df["EducationClassificationSummary"].astype(str).str.strip()
df["ED"] = df["LAP Economically Disadvantaged"].astype(str).str.strip()

def run_anova(factor_col, outcome_col):
    levels = df[factor_col].dropna().unique()
    groups = [df[df[factor_col] == lvl][outcome_col].dropna()
              for lvl in levels]
    if len(groups) >= 2:
        F, p = f_oneway(*groups)
        return {
            "factor": factor_col,
            "outcome": outcome_col,
            "levels": levels,
            "F": float(F),
            "p": float(p),
            "group_sizes": [len(g) for g in groups],
            "group_means": [g.mean() for g in groups]
        }
    else:
        return None

factors = ["NewRet", "Edu", "ED"]
outcomes = ["Math_A_num", "ELA_A_num", "ELA_Read_num", "ELA_Write_num"]

anova_results = {}
for f in factors:
    anova_results[f] = {}
    for o in outcomes:
        anova_results[f][o] = run_anova(f, o)

from pprint import pprint
pprint(anova_results)


In [ ]:
# --- ANOVA Analyis, question subtype ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.anova import AnovaRM

file_path = "2025_LEAP.xlsx"
sheet_name = "8th_Grade " # Change grade level

df = pd.read_excel(file_path, sheet_name=sheet_name)
print (df)

# Keep only the three reading subtype columns and code them 1/2/3
#    Weak = 1, Moderate = 2, Strong = 3
reading_cols = [
    "Reading Informational Text",
    "Reading Literary Text",
    "Reading Vocabulary"
]

likert_map = {"Weak": 1, "Moderate": 2, "Strong": 3}

for col in reading_cols:
    df[col + "_num"] = df[col].map(likert_map)

numeric_cols = [c + "_num" for c in reading_cols]

# Drop missing values
df_clean = df.dropna(subset=numeric_cols).reset_index(drop=True)
# Repeated-measures ANOVA
long_df = df_clean[numeric_cols].reset_index().melt(
    id_vars="index",
    value_vars=numeric_cols,
    var_name="Subtype",
    value_name="Score"
).rename(columns={"index": "Student"})

# Clean up subtype labels
long_df["Subtype"] = long_df["Subtype"].str.replace("_num", "", regex=False)

print("Mean score by reading subtype:")
print(long_df.groupby("Subtype")["Score"].mean())
print()

# Repeated-measures ANOVA
# Tests if mean performance differs across the 3 reading suptypes
anova_model = AnovaRM(
    data=long_df,
    depvar="Score",
    subject="Student",
    within=["Subtype"]
).fit()

print("Repeated-measures ANOVA on Reading Subtypes")
anova_table = anova_model.anova_table
print(anova_table)
print()
# Extract p-value (Pr > F) for Subtype
p_value = anova_table.loc["Subtype", "Pr > F"]
f_value = anova_table.loc["Subtype", "F Value"]

print(f"F-statistic for Subtype: {f_value:.4f}")
print(f"Pr > F (p-value):        {p_value:.4f}")
print()


# Interpretation
# 
alpha = 0.05  # significance level

if p_value < alpha:
    interpretation = (
        "There IS a statistically significant difference in performance "
        "across the three reading subtypes (p < 0.05). "
        "At least one subtype's mean performance differs from the others."
    )
else:
    interpretation = (
        "There is NO statistically significant difference in performance "
        "across the three reading subtypes (p ≥ 0.05). "
        "Any differences in the mean scores may be due to random variation."
    )

print("Interpretation:")
print(interpretation)